# Column cards

Column cards are the exact layer of the report: one dict per column,
computed over every row at any table size. No sampling touches them. The
card answers three questions — what role does the column play, what are
its summary statistics, and how concentrated are its values — plus a
format-signature block for strings. This chapter derives the rules behind
each answer.

## Exact counting at any cardinality

Every count on a card — distinct, nulls, top values, concentration — is
exact. The counting routine is adaptive over a compact value key: 64-bit
integers as-is, floats as bit patterns (NaN never stored, −0.0 normalized
to +0.0), strings, and booleans. It accumulates in a hash map until the
map holds 4,096 distinct keys; if the column has more, it switches to
sorting the compact keys and counting runs. The limit chooses the
*algorithm*, never the accuracy: both paths examine every row and produce
the exact distinct count.

Only the reported top values (top 64, ordered by count descending then key
ascending) own their string data; the high-cardinality sort borrows
strings. Concentration is computed over the full value distribution, not
over the top-k sample — the exactness of `effective_cardinality` on a
million-distinct column comes from the same pass that counted it. The
implementation notes live in
[the counting chapter](../implementation/counting.ipynb).

## Role inference

Roles are a rule-based reading of exact counts — deterministic, not a
classifier:

| Role | Rule |
|---|---|
| `all_null` | no non-null values |
| `constant` | one distinct value |
| `boolean`, `date`, `datetime` | boolean / date / datetime dtype |
| `categorical` (strings) | ≤ 500 distinct values |
| `free_text` (strings) | more than 500 distinct values |
| `categorical` (numeric) | ≤ 12 distinct with n ≥ 30, or distinct ratio ≤ 2% with ≤ 100 distinct and n ≥ 100 |
| `identifier` (numeric) | distinct ratio ≥ 0.98 |
| `continuous` (numeric) | otherwise |
| `other` | dtype the profiler treats as opaque (reported in coverage) |

Two cases need interpretation. A column of continuous measurements with no repeats — random floats, timestamps at full
precision — has distinct ratio 1.0 and reads as `identifier`. The rule is
about behavior, not semantics: such a column cannot be binned usefully
and behaves like a row label. And the numeric-categorical boundary is a
threshold like any other; a 13-level ordered code is still a code,
whatever the role says. Numeric columns inferred categorical carry
`treat_as_categorical` in the card's `action_hints` list; the finding
that says the same thing (`column_roles`) is a separate, `inferred`-basis
statement.

In [ ]:
import polars as pl

import onceover

n = 2_000
df = pl.DataFrame({
    "flag": [True, False] * (n // 2),
    "code": [i % 7 for i in range(n)],          # 7-level numeric code
    "tiny": [i % 2 for i in range(n)],           # 2-level numeric code
    "random_float": [i * 0.5 for i in range(n)],  # distinct ratio 1.0
})
report = onceover.profile(df, budget="quick")
for name in ["flag", "code", "tiny", "random_float"]:
    card = report.columns[name]
    print(f"{name:14s} role={card['role']:12s} distinct={card['distinct']:6d} "
          f"hints={card.get('action_hints', [])}")

## Concentration

The `concentration` block summarizes the value distribution with three
complementary measures, all exact:

- **HHI** — the Herfindahl–Hirschman index, the sum of squared shares.
  A constant scores 1.0; k equally frequent values score 1/k.
- **Shannon entropy** in nats, and its exponent, **effective
  cardinality** — "this column behaves like it has N distinct values".
  A six-level column dominated by one level scores near 1; a balanced
  six-level column scores near 6.
- **top-1 and top-3 shares** — the mass of the most frequent values.

Concentration findings use measured thresholds, and each finding states
the number that triggered it: a constant (`distinct ≤ 1`), a
near-constant (top-1 share ≥ 0.99), and a high-concentration flag when the
top value holds ≥ 90% of mass, reported with the effective cardinality.
A column can be concentrated without being flagged — the thresholds are
disclosure rules, not definitions.

## Format patterns

String cards carry `format_patterns`: shape signatures in which digits
become `#`, lowercase letters `a`, uppercase letters `A`, and other
characters are kept, truncated at 24 characters. Half the invoices shaped
`AAA-####` and half shaped `####` is a visible mixture — the card reports
each pattern and its share, and a `formats` finding is emitted when the
top pattern holds under 90% of values and the second pattern at least 5%.
The finding carries both shares and example values, so the mixture can be
inspected without pulling rows.

Holidays, currencies, and phone books are not modeled — the signatures are
shapes, not semantics. What they buy is cheap: mixed date formats, stray
padding, and half-migrated code systems are the most common "why doesn't
this join" causes, and they are visible in the signature distribution.